# Fine tune AG-NEWS Classification Model to IMDB Dataset With LORA

In [37]:
import torch
import math
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

#### **Dataset Loading**

In [38]:
dataset = load_dataset("AG_NEWS")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [39]:
dataset['train'][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [40]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [42]:
label_list = []

for idx,data in enumerate(dataset['train']):
    label_list.append(dataset['train'][idx]['label'])

num_classes = len(set(label_list))
num_classes

4

In [69]:
ag_news_labels = {0: "world", 1:"sports", 2:"business", 3:"sci/tech"}

#### **Define Tokenizer Function**

In [43]:
def tokenize_text (data):
    return tokenizer(data['text'], padding='max_length', truncation=True)


In [44]:
dataset = dataset.map(tokenize_text, batched=True)


In [45]:
dataset.column_names

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}

In [46]:
dataset = dataset.rename_column('label', 'labels')
dataset.set_format('torch')

In [47]:
dataset_train = dataset['train']
dataset_test = dataset['test']

In [67]:
dataset_train[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'labels': tensor(2),
 'input_ids': tensor([  101,  6250,  1457,   119, 10169,   140,  9598,  4388, 14000,  1103,
          2117,   113, 11336, 27603,   114, 11336, 27603,   118,  6373,   118,
         18275,  1116,   117,  6250,  1715,   112,   188,   173, 11129,  1979,
           165,  1467,  1104, 18737,   118,   172,  5730,  4724,   117,  1132,
          3195,  2448,  1254,   119,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

In [49]:
dataset_train[0]['labels']

tensor(2)

**Split train set**

In [50]:
train_split, valid_split = dataset_train.train_test_split(0.1, seed=42)

### **Define Dataloaders**

In [51]:
BATCH_SIZE = 64

In [52]:
train_dataloader = DataLoader(train_split, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_dataloader = DataLoader(valid_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### **Define Text-Embedding**

In [53]:
class TextEmbeddings(nn.Module):
    
    def __init__(self, vocab_size, embed_dims, dropout):
        super().__init__()
        
        self.embed_dim = embed_dims
        self.dropout = nn.Dropout(dropout)
        self.embeddings = nn.Embedding(vocab_size, embed_dims)
    
    
    def forward (self, input_tokens):
        # input_token -> [batch_size, seq_len]
        embed_out = self.embeddings(input_tokens) * math.sqrt(self.embed_dim)
        return self.dropout(embed_out)

### **Positional Embeddings**

In [54]:
class PositionalEmbedding (nn.Module):
    
    def __init__(self, vocab_size, d_model, max_seq_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        positions = torch.arange(0, max_seq_len).unsqueeze(1)
        
        pe = torch.zeros(max_seq_len, d_model).float()
        
        div_term = torch.exp(
            torch.arange(0,d_model,2) * (-math.log(10000)/d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe",pe)
        
    def forward(self, text_embeddings):
        
        text_embed_size = text_embeddings.size(1)
        pos_embed = text_embeddings + self.pe[:,0:text_embed_size,:]
        return self.dropout(pos_embed)
        

### **Define Model**

In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [62]:
class CustomClassificationModel(nn.Module):
    
    def __init__(self, vocab_size, d_model,n_classes ,n_layers, n_heads, dropout, dim_feedforward):
        super().__init__()
        
        self.embeddings = TextEmbeddings(vocab_size,d_model, dropout)
        self.pos_embeddings = PositionalEmbedding(vocab_size,d_model,2000, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model, dim_feedforward=dim_feedforward, 
                                                   batch_first=True, nhead=n_heads)
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, 
                                                         num_layers=n_layers)
        
        self.linear_layer = nn.Linear(d_model,n_classes)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_ids, attention_mask:None):
        # input_ids -> [batch_size, seq_len]
        
        embed_out = self.embeddings(input_ids) 
        pos_out = self.pos_embeddings(embed_out)
        
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        
        transformer_out = self.transformer_encoder(pos_out,src_key_padding_mask = src_key_padding_mask)
        
        out = transformer_out.mean(dim=1)
        
        out = self.linear_layer(out)
        
        return out
        

### **Define Model Instance** 

In [63]:
vocab_size = tokenizer.vocab_size
d_model = 512
num_layers = 6
num_heads = 8
dropout = 0.1
dim_feedforward = 2048

In [64]:
custom_model = CustomClassificationModel(vocab_size, d_model, num_classes, num_layers,
                                        num_heads, dropout, dim_feedforward)

In [68]:
custom_model.to(device)

CustomClassificationModel(
  (embeddings): TextEmbeddings(
    (dropout): Dropout(p=0.1, inplace=False)
    (embeddings): Embedding(28996, 512)
  )
  (pos_embeddings): PositionalEmbedding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (linear_layer): Linear(in_features=512, o

### **Define Model Prediction**

In [73]:
def model_predict (model:CustomClassificationModel, input:str):
    
    with torch.no_grad():
        tokens = tokenizer(input, return_tensors='pt').to(device)
        input_ids = tokens['input_ids']
        attention_mask = tokens['token_type_ids']
    
        output = model(input_ids,attention_mask)
        out = torch.argmax(output, dim=-1)
        
        predict_label = ag_news_labels[out.item()]
        
        return predict_label
    

In [74]:
sample = "The United Nations held an emergency meeting after rising tensions between neighboring countries sparked concerns about regional stability."

In [75]:
sample_predict_label = model_predict(custom_model,sample)
sample_predict_label

'sci/tech'